## Explainable Boosting Machine

In [ ]:
# Python ≥3.9 is required
import sys
assert sys.version_info >= (3, 9)

# Scikit-Learn ≥1.0 is required
import sklearn
assert sklearn.__version__ >= "1.0"

# Common imports
import numpy as np
import pandas as pd
import os

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Precision options
np.set_printoptions(precision=2)
pd.options.display.float_format = '{:.3f}'.format

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")
# ignore convergence warnings for the purpose of this class
warnings.filterwarnings(action="ignore", category=sklearn.exceptions.ConvergenceWarning)

In [ ]:
from google.colab import files
import os

In [ ]:
os.makedirs("data", exist_ok=True)

In [ ]:
uploaded = files.upload()

In [ ]:
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")

In [ ]:
train_import = pd.read_csv('data/train_bin_nhanes.csv')

In [ ]:
uploaded = files.upload()

In [ ]:
for filename in uploaded.keys():
    os.rename(filename, f"data/{filename}")

In [ ]:
test_import = pd.read_csv('data/test_bin_nhanes.csv')

In [ ]:
columns_to_remove = [
    "LBXGLU",
    "URDACT",
    "RIDAGEYR",
    "BMXBMI",
    "LBDHDD",
    "INDFMPIR",
    "LBXIN",
    "SLD012",
    "LBXTLG",
    "SLD013"
]

train_import = train_import.drop(columns=columns_to_remove)
test_import = test_import.drop(columns=columns_to_remove)

In [ ]:
X_train = train_import.drop(columns=["target"])
y_train = train_import["target"]

X_test = test_import.drop(columns=["target"])
y_test = test_import["target"]

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(y_train.unique())

In [ ]:
# Display scores function
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", f"{scores.mean():.2f}")
    print("Standard deviation:", f"{scores.std():.2f}")

In [ ]:
#reduced to 5 to reduce time
from sklearn.model_selection import cross_validate, KFold
n_splits = 5
k_fold = KFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

In [ ]:
!pip install interpret

In [ ]:
# add noise variables for extreme overfitting
for i in range(144):
    X_train[f"noise{i}"] = np.random.normal(size=len(X_train))
    X_test[f"noise{i}"] = np.random.normal(size=len(X_test))

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(y_train.unique())

In [ ]:
# Extreme EBM
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

ebm_extreme = ExplainableBoostingClassifier(
    max_bins=712,
    max_interaction_bins=456,
    interactions=350,
    learning_rate=0.001,
    max_rounds=1500,
    min_samples_leaf=1,
    outer_bags=1,
    inner_bags=0,
    random_state=77
)

ebm_extreme.fit(X_train, y_train)

# Training performance

y_train_pred_ebm_extreme = ebm_extreme.predict(X_train)

y_train_probs_ebm_extreme = (
    ebm_extreme.predict_proba(X_train)[:, 1]
)

train_auc_ebm_extreme = roc_auc_score(
    y_train,
    y_train_probs_ebm_extreme
)


# Hold-out performance

y_test_pred_ebm_extreme = ebm_extreme.predict(X_test)

y_test_probs_ebm_extreme = (
    ebm_extreme.predict_proba(X_test)[:, 1]
)

test_auc_ebm_extreme = roc_auc_score(
    y_test,
    y_test_probs_ebm_extreme
)



print("Extreme Overfitting")
print("Training ROC-AUC:", train_auc_ebm_extreme)
print("Hold-out ROC-AUC:", test_auc_ebm_extreme)

In [ ]:
from sklearn.utils import resample
import numpy as np

def bootstrap_optimism_auc(model, X_train, y_train, B=50):

    apparent_scores = []
    test_scores = []
    optimisms = []

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    for i in range(B):

        # Bootstrap sample
        X_boot, y_boot = resample(
            X_train,
            y_train,
            replace=True
        )

        # Fit model on bootstrap sample
        model.fit(X_boot, y_boot)

        # Bootstrap performance
        boot_probs = model.predict_proba(X_boot)[:, 1]

        # Performance on original training data
        full_probs = model.predict_proba(X_train)[:, 1]

        boot_auc = roc_auc_score(
            y_boot,
            boot_probs
        )

        test_auc = roc_auc_score(
            y_train,
            full_probs
        )

        optimism = boot_auc - test_auc

        apparent_scores.append(boot_auc)
        test_scores.append(test_auc)
        optimisms.append(optimism)

    avg_optimism = np.mean(optimisms)

    # Final model fitted to full training data
    model.fit(X_train, y_train)

    apparent_final = roc_auc_score(
        y_train,
        model.predict_proba(X_train)[:, 1]
    )

    corrected = apparent_final - avg_optimism

    return {
        "apparent": apparent_final,
        "optimism": avg_optimism,
        "corrected": corrected
    }

In [ ]:
# No EBM
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

ebm_moderate = ExplainableBoostingClassifier(
    max_bins=8,
    max_interaction_bins=4,
    interactions=0,
    learning_rate=0.00005,
    max_rounds=50,
    min_samples_leaf=5000,
    outer_bags=8,
    inner_bags=0,
    random_state=77
)

ebm_moderate.fit(X_train, y_train)


# Training performance

y_train_pred_ebm_moderate = ebm_moderate.predict(X_train)

y_train_probs_ebm_moderate = (
    ebm_moderate.predict_proba(X_train)[:, 1]
)

train_auc_ebm_moderate = roc_auc_score(
    y_train,
    y_train_probs_ebm_moderate
)


# Hold-out performance

y_test_pred_ebm_moderate = ebm_moderate.predict(X_test)

y_test_probs_ebm_moderate = (
    ebm_moderate.predict_proba(X_test)[:, 1]
)

test_auc_ebm_moderate = roc_auc_score(
    y_test,
    y_test_probs_ebm_moderate
)


print("No Overfitting")
print("Training ROC-AUC:", train_auc_ebm_moderate)
print("Hold-out ROC-AUC:", test_auc_ebm_moderate)

In [ ]:
# No EBM - Bootstrap

ebm_moderate_bootstrap = bootstrap_optimism_auc(
    ebm_moderate,
    X_train,
    y_train,
    B=50
)

print("No Overfitting - Bootstrap")
print("Apparent AUC:", ebm_moderate_bootstrap["apparent"])
print("Optimism:", ebm_moderate_bootstrap["optimism"])
print("Corrected AUC:", ebm_moderate_bootstrap["corrected"])